In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
import plotly.graph_objects as go

# ============================================================
# TVP-VAR GFEVD -> Connectedness + Pairwise NET
# ============================================================
# Assumption
# - theta[t, i, j]
# - row i = response variable
# - col j = shock source
#
# theta[target, source]
# = source -> target spillover
#
# pairwise_net[source, target]
# = theta[target, source] - theta[source, target]
#
# > 0 : source -> target dominance
# < 0 : target -> source dominance
# ============================================================

# ============================================================
# Config
# ============================================================

BASE_DIR = Path("./")

OUT_DIR = BASE_DIR / "connectedness_output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

GFEVD_FILE = BASE_DIR / "gfevd_all.npy"

DATE_FILE_CANDIDATES = [
    BASE_DIR / "tvpvar_input_scaled.csv",
    BASE_DIR / "tvpvar_input_transformed.csv",
    BASE_DIR / "tvpvar_raw_level_merged.csv",
]

DEFAULT_VAR_NAMES = [
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
    "dlog_DXY",
    "dlog_OIL",
    "dlog_GAS",
    "d_UST10Y",
    "d_VIX",
    "d_PJM_LMP",
]

SOLVPN_NAME = "dlog_SOLVPN"

SELECTED_PAIRS = [
    ("dlog_SOLVPN", var)
    for var in DEFAULT_VAR_NAMES
    if var != "dlog_SOLVPN"
]

SHOW_INTERACTIVE = True
SAVE_SELECTED_PAIR_PNG = True

EPS = 1e-12

# ============================================================
# Helper
# ============================================================

def infer_gfevd_tnn(arr):

    if arr.ndim != 3:
        raise ValueError(f"GFEVD must be 3D. Got {arr.shape}")

    if arr.shape[1] == arr.shape[2]:
        return arr.astype(float, copy=True)

    if arr.shape[0] == arr.shape[1]:
        return np.transpose(arr, (2, 0, 1)).astype(float, copy=True)

    raise ValueError(f"Cannot infer shape: {arr.shape}")


def infer_var_names(n):

    if len(DEFAULT_VAR_NAMES) == n:
        return DEFAULT_VAR_NAMES

    print("[WARN] Variable length mismatch.")
    return [f"VAR{i+1}" for i in range(n)]


def load_dates(target_len):

    for fp in DATE_FILE_CANDIDATES:

        if not fp.exists():
            continue

        try:
            df = pd.read_csv(fp)

            date_col = None

            for c in ["Date", "date", "DATE"]:
                if c in df.columns:
                    date_col = c
                    break

            if date_col is None:
                continue

            dt = pd.to_datetime(df[date_col], errors="coerce")
            dt = dt.dropna().reset_index(drop=True)

            if len(dt) < target_len:
                continue

            return dt.iloc[-target_len:].reset_index(drop=True)

        except Exception as e:
            print(f"[WARN] {fp}: {e}")

    return None


def normalize_gfevd_to_percent(arr):

    arr = np.asarray(arr, dtype=float).copy()

    if not np.all(np.isfinite(arr)):
        raise ValueError("NaN/inf in GFEVD.")

    arr[arr < 0] = 0.0

    row_sum = arr.sum(axis=2, keepdims=True)

    if np.any(row_sum <= EPS):
        raise ValueError("Zero row sum in GFEVD.")

    theta_pct = arr / row_sum * 100.0

    return theta_pct


def compute_connectedness(theta_pct):

    diag = np.diag(theta_pct)

    from_ = theta_pct.sum(axis=1) - diag
    to_ = theta_pct.sum(axis=0) - diag
    net_ = to_ - from_

    tci = (theta_pct.sum() - np.trace(theta_pct)) / theta_pct.shape[0]

    return from_, to_, net_, tci


def compute_pairwise_net(theta_pct):

    return theta_pct.T - theta_pct


def save_series_plot(df, cols, title, ylabel, out_path, x_col):

    plt.figure(figsize=(14, 5))

    x = pd.to_datetime(df[x_col]) if x_col == "Date" else df[x_col]

    for c in cols:

        label = c.split("_", 1)[1] if "_" in c else c

        plt.plot(
            x,
            df[c],
            linewidth=1.1,
            label=label
        )

    if "NET" in title:
        plt.axhline(0.0, linestyle="--", linewidth=1.0)

    plt.title(title)
    plt.xlabel(x_col)
    plt.ylabel(ylabel)

    plt.legend(
        ncol=3,
        fontsize=8
    )

    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


def save_heatmap_png(df_mean, out_path):

    values = df_mean.values
    vmax = np.nanmax(np.abs(values))

    fig, ax = plt.subplots(figsize=(10, 8))

    im = ax.imshow(
        values,
        cmap="coolwarm",
        vmin=-vmax,
        vmax=vmax
    )

    labels = list(df_mean.index)

    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))

    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)

    for i in range(len(labels)):
        for j in range(len(labels)):

            ax.text(
                j,
                i,
                f"{values[i, j]:.2f}",
                ha="center",
                va="center",
                fontsize=8
            )

    ax.set_title("Mean Pairwise NET Spillover")

    fig.colorbar(im, ax=ax)

    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


def save_pair_series_png(df_time, x_col, pair_col, out_path):

    x = pd.to_datetime(df_time[x_col]) if x_col == "Date" else df_time[x_col]

    plt.figure(figsize=(12, 5))

    plt.plot(
        x,
        df_time[pair_col],
        linewidth=1.3
    )

    plt.axhline(
        0.0,
        linestyle="--",
        linewidth=1.0
    )

    plt.title(pair_col)
    plt.xlabel(x_col)
    plt.ylabel("Pairwise NET (%)")

    plt.tight_layout()

    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


def show_pair_series_interactive(df_time, x_col, pair_cols, title):

    fig = go.Figure()

    for col in pair_cols:

        fig.add_trace(
            go.Scatter(
                x=df_time[x_col],
                y=df_time[col],
                mode="lines",
                name=col
            )
        )

    fig.add_hline(y=0.0)

    fig.update_layout(
        title=title,
        xaxis_title=x_col,
        yaxis_title="Pairwise NET (%)",
        height=600,
        hovermode="x unified"
    )

    fig.show()

# ============================================================
# Load GFEVD
# ============================================================

if not GFEVD_FILE.exists():
    raise FileNotFoundError(GFEVD_FILE.resolve())

raw_gfevd = np.load(GFEVD_FILE)

gfevd_tnn = infer_gfevd_tnn(raw_gfevd)

T_eff, N, N2 = gfevd_tnn.shape

if N != N2:
    raise ValueError(gfevd_tnn.shape)

VAR_NAMES = infer_var_names(N)

print("[INFO] GFEVD shape:", gfevd_tnn.shape)
print("[INFO] Variables:", VAR_NAMES)

# ============================================================
# Normalize
# ============================================================

theta_pct_all = normalize_gfevd_to_percent(gfevd_tnn)

print("[INFO] Row normalization complete.")

# ============================================================
# Dates
# ============================================================

dates = load_dates(T_eff)

if dates is not None:

    index_col = "Date"
    index_values = dates

else:

    index_col = "t"
    index_values = np.arange(T_eff)

print("[INFO] Index:", index_col)

# ============================================================
# Connectedness
# ============================================================

records = []

pairwise_net_all = np.zeros((T_eff, N, N))

pairwise_time_df = pd.DataFrame({
    index_col: index_values
})

for t in range(T_eff):

    theta = theta_pct_all[t]

    from_, to_, net_, tci = compute_connectedness(theta)

    pairwise_matrix = compute_pairwise_net(theta)

    pairwise_net_all[t] = pairwise_matrix

    row = {
        index_col:
        index_values.iloc[t]
        if index_col == "Date"
        else int(index_values[t])
    }

    for i, name in enumerate(VAR_NAMES):

        row[f"FROM_{name}"] = from_[i]
        row[f"TO_{name}"] = to_[i]
        row[f"NET_{name}"] = net_[i]

    row["TCI"] = tci

    records.append(row)

    for i, src in enumerate(VAR_NAMES):

        for j, dst in enumerate(VAR_NAMES):

            if i == j:
                continue

            col_name = f"{src}_to_{dst}"

            if col_name not in pairwise_time_df.columns:
                pairwise_time_df[col_name] = np.nan

            pairwise_time_df.loc[t, col_name] = pairwise_matrix[i, j]

df_time = pd.DataFrame(records)

# ============================================================
# Mean Summary
# ============================================================

rows_mean = []

for name in VAR_NAMES:

    rows_mean.append({
        "Variable": name,
        "FROM_mean": df_time[f"FROM_{name}"].mean(),
        "TO_mean": df_time[f"TO_{name}"].mean(),
        "NET_mean": df_time[f"NET_{name}"].mean(),
    })

df_mean = pd.DataFrame(rows_mean)

theta_mean = theta_pct_all.mean(axis=0)

from_mean, to_mean, net_mean, tci_mean = compute_connectedness(theta_mean)

df_connectedness_table = pd.DataFrame(
    theta_mean,
    index=VAR_NAMES,
    columns=VAR_NAMES
)

df_pairwise = pd.DataFrame(
    pairwise_net_all.mean(axis=0),
    index=VAR_NAMES,
    columns=VAR_NAMES
)

# ============================================================
# SOLVPN outgoing
# ============================================================

df_solvpn_to_all = None

if SOLVPN_NAME in VAR_NAMES:

    solvpn_idx = VAR_NAMES.index(SOLVPN_NAME)

    df_solvpn_to_all = pd.DataFrame({
        index_col: index_values
    })

    for target_idx, target_name in enumerate(VAR_NAMES):

        if target_idx == solvpn_idx:
            continue

        col_name = f"{SOLVPN_NAME}_to_{target_name}"

        df_solvpn_to_all[col_name] = theta_pct_all[:, target_idx, solvpn_idx]

# ============================================================
# Save CSV
# ============================================================

out_time = OUT_DIR / "connectedness_time.csv"
out_mean = OUT_DIR / "connectedness_mean.csv"
out_table = OUT_DIR / "connectedness_table_mean.csv"
out_pairwise = OUT_DIR / "pairwise_net_mean.csv"
out_pairwise_time = OUT_DIR / "pairwise_net_time_wide.csv"

df_time.to_csv(out_time, index=False)
df_mean.to_csv(out_mean, index=False)
df_connectedness_table.to_csv(out_table)
df_pairwise.to_csv(out_pairwise)
pairwise_time_df.to_csv(out_pairwise_time, index=False)

print("[INFO] Saved CSV outputs")

if df_solvpn_to_all is not None:

    out_solvpn = OUT_DIR / "solvpn_to_all_spillover.csv"

    df_solvpn_to_all.to_csv(
        out_solvpn,
        index=False
    )

# ============================================================
# Save plots
# ============================================================

x_col = index_col

out_tci_png = OUT_DIR / "tci_series.png"
out_from_png = OUT_DIR / "from_series.png"
out_to_png = OUT_DIR / "to_series.png"
out_net_png = OUT_DIR / "net_series.png"

plt.figure(figsize=(12, 4))

x = pd.to_datetime(df_time[x_col]) if x_col == "Date" else df_time[x_col]

plt.plot(x, df_time["TCI"], linewidth=1.5)

plt.title("Total Connectedness Index (TCI)")
plt.xlabel(x_col)
plt.ylabel("TCI (%)")

plt.tight_layout()

plt.savefig(out_tci_png, dpi=300, bbox_inches="tight")
plt.close()

from_cols = [f"FROM_{name}" for name in VAR_NAMES]
to_cols = [f"TO_{name}" for name in VAR_NAMES]
net_cols = [f"NET_{name}" for name in VAR_NAMES]

save_series_plot(
    df_time,
    from_cols,
    "Directional FROM",
    "Contribution (%)",
    out_from_png,
    x_col
)

save_series_plot(
    df_time,
    to_cols,
    "Directional TO",
    "Contribution (%)",
    out_to_png,
    x_col
)

save_series_plot(
    df_time,
    net_cols,
    "NET Directional Connectedness",
    "NET (%)",
    out_net_png,
    x_col
)

# ============================================================
# Heatmap
# ============================================================

out_heatmap = OUT_DIR / "pairwise_net_heatmap.png"

save_heatmap_png(
    df_pairwise,
    out_heatmap
)

# ============================================================
# Pair PNG
# ============================================================

if SAVE_SELECTED_PAIR_PNG:

    for src, dst in SELECTED_PAIRS:

        pair_col = f"{src}_to_{dst}"

        if pair_col not in pairwise_time_df.columns:
            continue

        out_png = OUT_DIR / f"{pair_col}.png"

        save_pair_series_png(
            pairwise_time_df,
            x_col,
            pair_col,
            out_png
        )

# ============================================================
# Interactive
# ============================================================

if SHOW_INTERACTIVE:

    valid_pair_cols = []

    for src, dst in SELECTED_PAIRS:

        pair_col = f"{src}_to_{dst}"

        if pair_col in pairwise_time_df.columns:
            valid_pair_cols.append(pair_col)

    if valid_pair_cols:

        show_pair_series_interactive(
            pairwise_time_df,
            x_col,
            valid_pair_cols,
            "Interactive Pairwise NET Spillover"
        )

# ============================================================
# Diagnostics
# ============================================================

print("\n[INFO] Average TCI")
print(f"TCI_mean = {df_time['TCI'].mean():.4f}%")

print("\n[INFO] Pairwise mean shape")
print(df_pairwise.shape)

print("\n[INFO] Saved to")
print(OUT_DIR.resolve())

[INFO] GFEVD shape: (605, 10, 10)
[INFO] Variables: ['dlog_SOLVPN', 'dlog_COPPER', 'dlog_GOLD', 'dlog_SILVER', 'dlog_DXY', 'dlog_OIL', 'dlog_GAS', 'd_UST10Y', 'd_VIX', 'd_PJM_LMP']
[INFO] Row normalization complete.
[INFO] Index: Date
[INFO] Saved CSV outputs



[INFO] Average TCI
TCI_mean = 57.1258%

[INFO] Pairwise mean shape
(10, 10)

[INFO] Saved to
D:\University\3-2.5\PADA_Lab\DC_and_meterials_2\07_Connectedness\connectedness_output
